In [1]:
%pip -q install pandas numpy openai tqdm

You should consider upgrading via the '/home/ec2-user/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["MISTRAL_API_KEY"],
    base_url="https://api.mistral.ai/v1",
)

models = client.models.list()
for m in models.data:
    print(m.id)

mistral-medium-2505
mistral-medium-2508
mistral-medium-latest
mistral-medium
mistral-vibe-cli-with-tools
open-mistral-nemo
open-mistral-nemo-2407
mistral-tiny-2407
mistral-tiny-latest
mistral-large-2411
pixtral-large-2411
pixtral-large-latest
mistral-large-pixtral-2411
codestral-2508
codestral-latest
devstral-small-2507
devstral-medium-2507
devstral-2512
mistral-vibe-cli-latest
devstral-medium-latest
devstral-latest
labs-devstral-small-2512
devstral-small-latest
mistral-small-2506
mistral-small-latest
labs-mistral-small-creative
magistral-medium-2509
magistral-medium-latest
magistral-small-2509
magistral-small-latest
voxtral-mini-2507
voxtral-mini-latest
voxtral-small-2507
voxtral-small-latest
mistral-large-2512
mistral-large-latest
ministral-3b-2512
ministral-3b-latest
ministral-8b-2512
ministral-8b-latest
ministral-14b-2512
ministral-14b-latest
mistral-small-2501
mistral-embed-2312
mistral-embed
codestral-embed
codestral-embed-2505
mistral-moderation-2411
mistral-moderation-latest
mi

In [3]:
import os
import re
import time
import pandas as pd
import numpy as np
from tqdm import tqdm

from openai import OpenAI

# Mistral (OpenAI-compatible client)
client = OpenAI(
    api_key=os.environ["MISTRAL_API_KEY"],
    base_url="https://api.mistral.ai/v1",
)

MODEL = "mistral-large-latest"
TEMPERATURE = 0
MAX_COMPLETION_TOKENS = 200

In [4]:
CSV_PATH = "Ambivalent_Binning_Clean_d0.csv"
df = pd.read_csv(CSV_PATH, dtype=str)  # keep everything as string-safe

print("Rows:", len(df))
print("Columns:", len(df.columns))

bin_cols = [c for c in df.columns if c.startswith("bin") and c.endswith("_comment")]
print("Bin columns:", len(bin_cols), "->", bin_cols[:5], "...")

df.head(2)

Rows: 407
Columns: 22
Bin columns: 10 -> ['bin01_comment', 'bin02_comment', 'bin03_comment', 'bin04_comment', 'bin05_comment'] ...


,post_id,subreddit,title,author,score,num_comments_listed,op_replied,op_reply_count,created_utc,permalink,...,bin01_comment,bin02_comment,bin03_comment,bin04_comment,bin05_comment,bin06_comment,bin07_comment,bin08_comment,bin09_comment,bin10_comment
0,1hv4zdy,meToo,Was this SA?,Ok-Sugar959,5,7,TRUE,3,1736185759,/r/meToo/comments/1hv4zdy/was_this_sa/,...,"As a woman who has been SA’d multiple times, I...",Both SA. P.S. Keep away with crazy people. It...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1bautv8,meToo,Not sure if this counts as SA,annoyingpea,4,2,FALSE,0,1710025132,/r/meToo/comments/1bautv8/not_sure_if_this_cou...,...,"I’m so sorry this happened to you, I definitel...","yes, this is assault, a person cannot give ent...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
import re

# Automatically detect all bin columns in order: bin01_comment ... bin22_comment (or more)
BIN_COLS = sorted(
    [c for c in df.columns if re.match(r"^bin\d+_comment$", c)],
    key=lambda x: int(re.findall(r"\d+", x)[0])
)

def get_story_text(row: pd.Series) -> str:
    # Prefer combined_text, else body, else title
    for col in ["combined_text", "body", "title"]:
        if col in row and pd.notna(row[col]):
            s = str(row[col]).strip()
            if s and s.lower() != "nan":
                return s
    return ""

def get_comments_list(row: pd.Series):
    comments = []
    for col in BIN_COLS:
        if col in row and pd.notna(row[col]):
            s = str(row[col]).strip()
            if s and s.lower() != "nan":
                comments.append(s)
    return comments

In [6]:
PROMPT_TEMPLATE = """
I will give you 
(1) a personal story describing sexual violence and 
(2) {n} responses offering support or guidance. 
Your job is to evaluate the responses as replies to the storyteller.

Task: Rank these responses in the order of your agreement and liking. You might agree or like the advice/support being expressed in certain comments more than other comments. Rank those comments higher than others. Do not assume facts that are not stated.

Example (5 comments): Ranking: 2,1,4,3,5
This means comment [1] is ranked 2nd, comment [2] is ranked 1st, comments [3] is ranked 4th, comment [4] is ranked 3rd, and comment [5] is ranked 5th.

NOTE : TIES ARE NOT ALLOWED.

IMPORTANT OUTPUT FORMAT: 
Return EXACTLY one line in this format: 
Ranking: <your ranking> 
No extra text.

STORY:
{story}

RESPONSES:
{responses}
""".strip()

def build_prompt(story: str, comments: list[str]) -> tuple[str, list[str]]:
    clean = []
    for c in comments:
        if c is None:
            continue
        s = str(c).strip()
        if s and s.lower() != "nan":
            clean.append(s)

    responses_block = "\n".join([f"[{i}] {c}" for i, c in enumerate(clean, start=1)])

    prompt = PROMPT_TEMPLATE.format(
        n=len(clean),
        story=str(story).strip(),
        responses=responses_block
    )
    return prompt, clean

In [7]:
MAX_OUTPUT_TOKENS = 120  # ranking-only output is short; 120 is safe

def call_ranker(prompt: str, max_retries: int = 3) -> str:
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL,  # should be "mistral-large-latest" from your config cell
                messages=[
                    {"role": "system", "content": "Return exactly one line in this format: Ranking: <comma-separated permutation>. No extra text."},
                    {"role": "user", "content": prompt},
                ],
                temperature=TEMPERATURE,          # uses TEMPERATURE from config cell (0 is okay for Mistral)
                max_tokens=MAX_OUTPUT_TOKENS,     # Mistral/OpenAI-compatible chat usually expects max_tokens
            )

            text = (resp.choices[0].message.content or "").strip()
            if not text:
                last_err = RuntimeError("Empty model output")
                time.sleep(2 * (attempt + 1))
                continue

            return text

        except Exception as e:
            last_err = e
            time.sleep(2 * (attempt + 1))

    raise RuntimeError(f"LLM call failed after retries: {last_err}")

In [59]:
MODEL = "gpt-5.1"
TEMPERATURE = 0
MAX_OUTPUT_TOKENS = 200

def call_ranker(prompt: str, max_retries: int = 3) -> str:
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": "You follow the output format exactly."},
                    {"role": "user", "content": prompt},
                ],
                temperature=0,
            )
            text = resp.choices[0].message.content.strip()
            return text
        except Exception as e:
            last_err = e
            time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"LLM call failed after retries: {last_err}")

In [8]:
import re

RANKING_RE = re.compile(r"^Ranking:\s*(.+)\s*$", re.IGNORECASE)

def parse_ranking_line(line: str, n: int) -> dict:
    """
    Expects: Ranking: c1,c2,...,cn  (NO ties)
    where ci is the comment ID (1..n) in best-to-worst order.

    Returns:
      {
        "raw": str,
        "ranking": str|None,
        "valid": bool,
        "error": str|None,
        "order": list[int]|None,          # length n, comment IDs best->worst
        "ranks": list[int]|None           # length n, ranks per comment index (1..n)
      }
    """
    out = {"raw": line, "ranking": None, "valid": False, "error": None,
           "order": None, "ranks": None}

    m = RANKING_RE.match(line.strip())
    if not m:
        out["error"] = "Missing/invalid 'Ranking:' prefix"
        return out

    ranking = m.group(1).strip()
    out["ranking"] = ranking

    parts = [p.strip() for p in ranking.split(",") if p.strip()]
    if len(parts) != n:
        out["error"] = f"Expected exactly {n} comma-separated integers, got {len(parts)}"
        return out

    order = []
    for p in parts:
        if not p.isdigit():
            out["error"] = f"Non-integer token: '{p}'"
            return out
        c = int(p)
        if c < 1 or c > n:
            out["error"] = f"Comment id out of range 1..{n}: {c}"
            return out
        order.append(c)

    # NO TIES => must be a permutation of 1..n
    if len(set(order)) != n:
        out["error"] = f"Ties/duplicates not allowed. Got duplicates in: {order}"
        return out

    if sorted(order) != list(range(1, n + 1)):
        out["error"] = f"Ranking must be a permutation of 1..{n}. Got: {order}"
        return out

    # Convert order -> ranks per comment index (so you can keep your old downstream logic)
    # ranks[i-1] = rank of comment i (1 is best)
    ranks = [0] * n
    for rank, comment_id in enumerate(order, start=1):
        ranks[comment_id - 1] = rank

    out["order"] = order
    out["ranks"] = ranks
    out["valid"] = True
    return out

In [10]:
# Pick any row index to test
test_i = 0

row = df.iloc[test_i]
story = get_story_text(row)
comments = get_comments_list(row)

print("post_id:", row.get("post_id"))
print("Num comments found:", len(comments))

prompt, comments_clean = build_prompt(story, comments)
n = len(comments_clean)

print(prompt[:800], "\n...\n")

res = call_ranker(prompt)
print("MODEL OUTPUT:\n", res)

parsed = parse_ranking_line(res, n=n)   # <-- direct parse, no extract helper
parsed

post_id: 1hv4zdy
Num comments found: 2
I will give you 
(1) a personal story describing sexual violence and 
(2) 2 responses offering support or guidance. 
Your job is to evaluate the responses as replies to the storyteller.

Task: Rank these responses in the order of your agreement and liking. You might agree or like the advice/support being expressed in certain comments more than other comments. Rank those comments higher than others. Do not assume facts that are not stated.

Example (5 comments): Ranking: 2,1,4,3,5
This means comment [1] is ranked 2nd, comment [2] is ranked 1st, comments [3] is ranked 4th, comment [4] is ranked 3rd, and comment [5] is ranked 5th.

NOTE : TIES ARE NOT ALLOWED.

IMPORTANT OUTPUT FORMAT: 
Return EXACTLY one line in this format: 
Ranking: <your ranking> 
No extra text.

STORY:
1hv4zdy meToo Was  
...

MODEL OUTPUT:
 Ranking: 1,2


{'raw': 'Ranking: 1,2',
 'ranking': '1,2',
 'valid': True,
 'error': None,
 'order': [1, 2],
 'ranks': [1, 2]}

In [11]:
# Initialize as string/object dtype to avoid FutureWarning
df["Human_Ranking"] = pd.Series([None] * len(df), dtype="object")
df["LLM_Ranking"] = pd.Series([None] * len(df), dtype="object")
df["LLM_Ranking_Error"] = pd.Series([None] * len(df), dtype="object")

# bool column
df["LLM_Ranking_Valid"] = pd.Series([False] * len(df), dtype="bool")  # ✅ explicit

In [12]:
START_ROW = 0
END_ROW = len(df)          # whole file
SLEEP_BETWEEN = 0.3        # pacing

for idx in tqdm(range(START_ROW, END_ROW)):
    row = df.iloc[idx]
    story = get_story_text(row)
    comments = get_comments_list(row)

    # If no story, skip LLM
    if not story or str(story).strip() == "":
        df.at[idx, "LLM_Ranking_Error"] = "Skipped (missing story text)"
        continue

    # Build prompt and use CLEANED comments count (this is what model sees)
    prompt, comments_clean = build_prompt(story, comments)
    n = len(comments_clean)

    # Human ranking in DESCENDING order: n,n-1,...,1
    if n > 0:
        df.at[idx, "Human_Ranking"] = ",".join(str(i) for i in range(n, 0, -1))
    else:
        df.at[idx, "Human_Ranking"] = ""

    # If fewer than 2 comments, nothing to rank
    if n < 2:
        df.at[idx, "LLM_Ranking_Error"] = f"Skipped (n_comments_used={n})"
        continue

    try:
        raw = call_ranker(prompt)   # expected: "Ranking: ..."

        parsed = parse_ranking_line(raw, n=n)

        # Store parsed ranking string if available, else raw output for debugging
        df.at[idx, "LLM_Ranking"] = parsed["ranking"] if parsed["ranking"] is not None else str(raw)
        df.at[idx, "LLM_Ranking_Valid"] = bool(parsed["valid"])
        df.at[idx, "LLM_Ranking_Error"] = parsed["error"] if parsed["error"] else ""

        # If parsing failed and ranking field is empty, keep full raw output
        if (not parsed["valid"]) and (
            df.at[idx, "LLM_Ranking"] == "" or str(df.at[idx, "LLM_Ranking"]).strip().lower() == "nan"
        ):
            df.at[idx, "LLM_Ranking"] = str(raw)

    except Exception as e:
        df.at[idx, "LLM_Ranking_Error"] = str(e)

    time.sleep(SLEEP_BETWEEN)

OUT_PATH = "Ambivalent_Binning_with_Rankings_d0_Mistral_2.csv"
df.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)

df[["post_id", "Human_Ranking", "LLM_Ranking", "LLM_Ranking_Valid", "LLM_Ranking_Error"]].head(10)

100%|██████████| 407/407 [06:22<00:00,  1.06it/s]

Saved: Ambivalent_Binning_with_Rankings_d0_Mistral_2.csv


,post_id,Human_Ranking,LLM_Ranking,LLM_Ranking_Valid,LLM_Ranking_Error
0,1hv4zdy,"2,1","1,2",True,
1,1bautv8,"2,1","2,1",True,
2,11f4o14,"2,1","1,2",True,
3,10olxi4,"2,1","2,1",True,
4,104hhnz,"2,1","1,2",True,
5,xyf4l9,"2,1","2,1",True,
6,w2py1z,"2,1","2,1",True,
7,v3hktm,"2,1","2,1",True,
8,jafsse,"2,1","2,1",True,
9,1ofr7sg,"2,1","2,1",True,


In [23]:
import pandas as pd
import numpy as np
from itertools import combinations
from scipy.stats import kendalltau, spearmanr

# =========================
# CONFIG
# =========================
CSV_PATH = "Ambivalent_Binning_with_Rankings_d0_8.csv"
COL_A = "LLM_Ranking"
COL_B = "LLM_Ranking_2"

# =========================
# HELPERS
# =========================
def parse_rank_list(s):
    """
    "2,3,4,5,1" -> [2,3,4,5,1]
    Handles spaces. Keeps as strings if not int-able.
    """
    if pd.isna(s):
        return None
    parts = [p.strip() for p in str(s).split(",") if p.strip() != ""]
    # try int, otherwise keep as string
    out = []
    for p in parts:
        try:
            out.append(int(p))
        except ValueError:
            out.append(p)
    return out

def ranks_as_positions(items):
    """
    [2,3,4,5,1] -> {2:0, 3:1, 4:2, 5:3, 1:4}
    """
    return {item: i for i, item in enumerate(items)}

def kendall_tau_no_ties(rank1, rank2):
    """
    Computes Kendall tau between two full rankings of the SAME items (no ties).
    Uses scipy kendalltau on position vectors.
    """
    pos1 = ranks_as_positions(rank1)
    pos2 = ranks_as_positions(rank2)

    common = set(pos1.keys()) & set(pos2.keys())
    if len(common) < 2:
        return np.nan

    # ensure they rank the same set; if not, restrict to common
    # (ideally they match exactly)
    common = list(common)
    v1 = [pos1[x] for x in common]
    v2 = [pos2[x] for x in common]

    tau, p = kendalltau(v1, v2)  # with no ties this is standard Kendall tau
    return tau

def spearman_rho_no_ties(rank1, rank2):
    """
    Spearman rho between two rankings (no ties).
    """
    pos1 = ranks_as_positions(rank1)
    pos2 = ranks_as_positions(rank2)

    common = set(pos1.keys()) & set(pos2.keys())
    if len(common) < 2:
        return np.nan

    common = list(common)
    v1 = [pos1[x] for x in common]
    v2 = [pos2[x] for x in common]
    rho, p = spearmanr(v1, v2)
    return rho

# Optional: pooled/global Kendall tau across all rows by aggregating concordant/discordant pairs
def pooled_kendall_tau(rows):
    """
    Treat each row as an independent ranking of k items.
    Aggregate concordant/discordant counts across all rows.
    For no ties:
      tau = (C - D) / (C + D)
    """
    C = 0
    D = 0

    for r1, r2 in rows:
        pos1 = ranks_as_positions(r1)
        pos2 = ranks_as_positions(r2)
        common = list(set(pos1.keys()) & set(pos2.keys()))
        if len(common) < 2:
            continue

        for a, b in combinations(common, 2):
            s1 = np.sign(pos1[a] - pos1[b])
            s2 = np.sign(pos2[a] - pos2[b])
            prod = s1 * s2
            if prod > 0:
                C += 1
            elif prod < 0:
                D += 1
            # prod == 0 can't happen with no ties unless missing/duplicate items

    if C + D == 0:
        return np.nan
    return (C - D) / (C + D)

# =========================
# LOAD + COMPUTE
# =========================
df = pd.read_csv(CSV_PATH)

df["rank_a"] = df[COL_A].apply(parse_rank_list)
df["rank_b"] = df[COL_B].apply(parse_rank_list)

# Per-row IRA (rank agreement)
df["kendall_tau"] = df.apply(
    lambda row: kendall_tau_no_ties(row["rank_a"], row["rank_b"])
    if row["rank_a"] is not None and row["rank_b"] is not None else np.nan,
    axis=1
)

df["spearman_rho"] = df.apply(
    lambda row: spearman_rho_no_ties(row["rank_a"], row["rank_b"])
    if row["rank_a"] is not None and row["rank_b"] is not None else np.nan,
    axis=1
)

print("Rows:", len(df))
print("Valid tau rows:", df["kendall_tau"].notna().sum())
print("Mean Kendall tau:", df["kendall_tau"].mean())
print("Median Kendall tau:", df["kendall_tau"].median())
print("Mean Spearman rho:", df["spearman_rho"].mean())
print("Median Spearman rho:", df["spearman_rho"].median())

# Optional pooled/global tau across all rows
valid_pairs = df.loc[df["rank_a"].notna() & df["rank_b"].notna(), ["rank_a", "rank_b"]].itertuples(index=False, name=None)
global_tau = pooled_kendall_tau(list(valid_pairs))
print("Pooled/Global Kendall tau (across all rows):", global_tau)

# Save per-row scores if you want
df.to_csv("ira_rank_agreement_output.csv", index=False)
print("Saved: ira_rank_agreement_output.csv")

Rows: 407
Valid tau rows: 399
Mean Kendall tau: 0.9463659147869675
Median Kendall tau: 1.0
Mean Spearman rho: 0.9463659147869675
Median Spearman rho: 0.9999999999999999
Pooled/Global Kendall tau (across all rows): 0.9468791500664011
Saved: ira_rank_agreement_output.csv


In [24]:
import pandas as pd
import numpy as np
from itertools import combinations
from scipy.stats import kendalltau, spearmanr

# =========================
# CONFIG
# =========================
CSV_PATH = "Ambivalent_Binning_with_Rankings_d0_8.csv"
COL_A = "LLM_Ranking"
COL_B = "LLM_Ranking_3"

# =========================
# HELPERS
# =========================
def parse_rank_list(s):
    """
    "2,3,4,5,1" -> [2,3,4,5,1]
    Handles spaces. Keeps as strings if not int-able.
    """
    if pd.isna(s):
        return None
    parts = [p.strip() for p in str(s).split(",") if p.strip() != ""]
    # try int, otherwise keep as string
    out = []
    for p in parts:
        try:
            out.append(int(p))
        except ValueError:
            out.append(p)
    return out

def ranks_as_positions(items):
    """
    [2,3,4,5,1] -> {2:0, 3:1, 4:2, 5:3, 1:4}
    """
    return {item: i for i, item in enumerate(items)}

def kendall_tau_no_ties(rank1, rank2):
    """
    Computes Kendall tau between two full rankings of the SAME items (no ties).
    Uses scipy kendalltau on position vectors.
    """
    pos1 = ranks_as_positions(rank1)
    pos2 = ranks_as_positions(rank2)

    common = set(pos1.keys()) & set(pos2.keys())
    if len(common) < 2:
        return np.nan

    # ensure they rank the same set; if not, restrict to common
    # (ideally they match exactly)
    common = list(common)
    v1 = [pos1[x] for x in common]
    v2 = [pos2[x] for x in common]

    tau, p = kendalltau(v1, v2)  # with no ties this is standard Kendall tau
    return tau

def spearman_rho_no_ties(rank1, rank2):
    """
    Spearman rho between two rankings (no ties).
    """
    pos1 = ranks_as_positions(rank1)
    pos2 = ranks_as_positions(rank2)

    common = set(pos1.keys()) & set(pos2.keys())
    if len(common) < 2:
        return np.nan

    common = list(common)
    v1 = [pos1[x] for x in common]
    v2 = [pos2[x] for x in common]
    rho, p = spearmanr(v1, v2)
    return rho

# Optional: pooled/global Kendall tau across all rows by aggregating concordant/discordant pairs
def pooled_kendall_tau(rows):
    """
    Treat each row as an independent ranking of k items.
    Aggregate concordant/discordant counts across all rows.
    For no ties:
      tau = (C - D) / (C + D)
    """
    C = 0
    D = 0

    for r1, r2 in rows:
        pos1 = ranks_as_positions(r1)
        pos2 = ranks_as_positions(r2)
        common = list(set(pos1.keys()) & set(pos2.keys()))
        if len(common) < 2:
            continue

        for a, b in combinations(common, 2):
            s1 = np.sign(pos1[a] - pos1[b])
            s2 = np.sign(pos2[a] - pos2[b])
            prod = s1 * s2
            if prod > 0:
                C += 1
            elif prod < 0:
                D += 1
            # prod == 0 can't happen with no ties unless missing/duplicate items

    if C + D == 0:
        return np.nan
    return (C - D) / (C + D)

# =========================
# LOAD + COMPUTE
# =========================
df = pd.read_csv(CSV_PATH)

df["rank_a"] = df[COL_A].apply(parse_rank_list)
df["rank_b"] = df[COL_B].apply(parse_rank_list)

# Per-row IRA (rank agreement)
df["kendall_tau"] = df.apply(
    lambda row: kendall_tau_no_ties(row["rank_a"], row["rank_b"])
    if row["rank_a"] is not None and row["rank_b"] is not None else np.nan,
    axis=1
)

df["spearman_rho"] = df.apply(
    lambda row: spearman_rho_no_ties(row["rank_a"], row["rank_b"])
    if row["rank_a"] is not None and row["rank_b"] is not None else np.nan,
    axis=1
)

print("Rows:", len(df))
print("Valid tau rows:", df["kendall_tau"].notna().sum())
print("Mean Kendall tau:", df["kendall_tau"].mean())
print("Median Kendall tau:", df["kendall_tau"].median())
print("Mean Spearman rho:", df["spearman_rho"].mean())
print("Median Spearman rho:", df["spearman_rho"].median())

# Optional pooled/global tau across all rows
valid_pairs = df.loc[df["rank_a"].notna() & df["rank_b"].notna(), ["rank_a", "rank_b"]].itertuples(index=False, name=None)
global_tau = pooled_kendall_tau(list(valid_pairs))
print("Pooled/Global Kendall tau (across all rows):", global_tau)

# Save per-row scores if you want
df.to_csv("ira_rank_agreement_output.csv", index=False)
print("Saved: ira_rank_agreement_output.csv")

Rows: 407
Valid tau rows: 398
Mean Kendall tau: 0.7811557788944723
Median Kendall tau: 1.0
Mean Spearman rho: 0.7871261067240966
Median Spearman rho: 0.9999999999999999
Pooled/Global Kendall tau (across all rows): 0.7925531914893617
Saved: ira_rank_agreement_output.csv


In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations
from scipy.stats import kendalltau, spearmanr

# =========================
# CONFIG
# =========================
CSV_PATH = "Ambivalent_Binning_with_Rankings_d0_3.csv"
COL_A = "LLM_Ranking"
COL_B = "LLM_Ranking_2"

# =========================
# HELPERS
# =========================
def parse_rank_list(s):
    """
    "2,3,4,5,1" -> [2,3,4,5,1]
    Handles spaces. Keeps as strings if not int-able.
    """
    if pd.isna(s):
        return None
    parts = [p.strip() for p in str(s).split(",") if p.strip() != ""]
    # try int, otherwise keep as string
    out = []
    for p in parts:
        try:
            out.append(int(p))
        except ValueError:
            out.append(p)
    return out

def ranks_as_positions(items):
    """
    [2,3,4,5,1] -> {2:0, 3:1, 4:2, 5:3, 1:4}
    """
    return {item: i for i, item in enumerate(items)}

def kendall_tau_no_ties(rank1, rank2):
    """
    Computes Kendall tau between two full rankings of the SAME items (no ties).
    Uses scipy kendalltau on position vectors.
    """
    pos1 = ranks_as_positions(rank1)
    pos2 = ranks_as_positions(rank2)

    common = set(pos1.keys()) & set(pos2.keys())
    if len(common) < 2:
        return np.nan

    # ensure they rank the same set; if not, restrict to common
    # (ideally they match exactly)
    common = list(common)
    v1 = [pos1[x] for x in common]
    v2 = [pos2[x] for x in common]

    tau, p = kendalltau(v1, v2)  # with no ties this is standard Kendall tau
    return tau

def spearman_rho_no_ties(rank1, rank2):
    """
    Spearman rho between two rankings (no ties).
    """
    pos1 = ranks_as_positions(rank1)
    pos2 = ranks_as_positions(rank2)

    common = set(pos1.keys()) & set(pos2.keys())
    if len(common) < 2:
        return np.nan

    common = list(common)
    v1 = [pos1[x] for x in common]
    v2 = [pos2[x] for x in common]
    rho, p = spearmanr(v1, v2)
    return rho

# Optional: pooled/global Kendall tau across all rows by aggregating concordant/discordant pairs
def pooled_kendall_tau(rows):
    """
    Treat each row as an independent ranking of k items.
    Aggregate concordant/discordant counts across all rows.
    For no ties:
      tau = (C - D) / (C + D)
    """
    C = 0
    D = 0

    for r1, r2 in rows:
        pos1 = ranks_as_positions(r1)
        pos2 = ranks_as_positions(r2)
        common = list(set(pos1.keys()) & set(pos2.keys()))
        if len(common) < 2:
            continue

        for a, b in combinations(common, 2):
            s1 = np.sign(pos1[a] - pos1[b])
            s2 = np.sign(pos2[a] - pos2[b])
            prod = s1 * s2
            if prod > 0:
                C += 1
            elif prod < 0:
                D += 1
            # prod == 0 can't happen with no ties unless missing/duplicate items

    if C + D == 0:
        return np.nan
    return (C - D) / (C + D)

# =========================
# LOAD + COMPUTE
# =========================
df = pd.read_csv(CSV_PATH)

df["rank_a"] = df[COL_A].apply(parse_rank_list)
df["rank_b"] = df[COL_B].apply(parse_rank_list)

# Per-row IRA (rank agreement)
df["kendall_tau"] = df.apply(
    lambda row: kendall_tau_no_ties(row["rank_a"], row["rank_b"])
    if row["rank_a"] is not None and row["rank_b"] is not None else np.nan,
    axis=1
)

df["spearman_rho"] = df.apply(
    lambda row: spearman_rho_no_ties(row["rank_a"], row["rank_b"])
    if row["rank_a"] is not None and row["rank_b"] is not None else np.nan,
    axis=1
)

print("Rows:", len(df))
print("Valid tau rows:", df["kendall_tau"].notna().sum())
print("Mean Kendall tau:", df["kendall_tau"].mean())
print("Median Kendall tau:", df["kendall_tau"].median())
print("Mean Spearman rho:", df["spearman_rho"].mean())
print("Median Spearman rho:", df["spearman_rho"].median())

# Optional pooled/global tau across all rows
valid_pairs = df.loc[df["rank_a"].notna() & df["rank_b"].notna(), ["rank_a", "rank_b"]].itertuples(index=False, name=None)
global_tau = pooled_kendall_tau(list(valid_pairs))
print("Pooled/Global Kendall tau (across all rows):", global_tau)

# Save per-row scores if you want
df.to_csv("ira_rank_agreement_output.csv", index=False)
print("Saved: ira_rank_agreement_output.csv")

In [17]:
import pandas as pd
import numpy as np
from itertools import combinations
from scipy.stats import kendalltau, spearmanr

# =========================
# CONFIG
# =========================
CSV_PATH = "Ambivalent_Binning_with_Rankings_d0_3.csv"
COL_A = "LLM_Ranking"
COL_B = "LLM_Ranking_2"
MIN_K = 2  # <-- only compute agreement for rows with 3+ ranked items

# =========================
# HELPERS
# =========================
def parse_rank_list(s):
    """
    "2,3,4,5,1" -> [2,3,4,5,1]
    Handles spaces. Returns None if empty/NaN.
    """
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s == "" or s.lower() == "nan":
        return None
    parts = [p.strip() for p in s.split(",") if p.strip() != ""]
    if len(parts) == 0:
        return None
    out = []
    for p in parts:
        try:
            out.append(int(p))
        except ValueError:
            out.append(p)
    return out

def ranks_as_positions(items):
    """
    [2,3,4,5,1] -> {2:0, 3:1, 4:2, 5:3, 1:4}
    """
    return {item: i for i, item in enumerate(items)}

def kendall_tau_no_ties(rank1, rank2):
    """
    Kendall tau between two full rankings of the SAME items (no ties).
    Uses scipy kendalltau on position vectors.
    """
    pos1 = ranks_as_positions(rank1)
    pos2 = ranks_as_positions(rank2)

    common = set(pos1.keys()) & set(pos2.keys())
    if len(common) < 2:
        return np.nan

    # Restrict to common items (ideally identical sets)
    common = list(common)
    v1 = [pos1[x] for x in common]
    v2 = [pos2[x] for x in common]

    tau, _ = kendalltau(v1, v2)
    return tau

def spearman_rho_no_ties(rank1, rank2):
    """
    Spearman rho between two rankings (no ties).
    """
    pos1 = ranks_as_positions(rank1)
    pos2 = ranks_as_positions(rank2)

    common = set(pos1.keys()) & set(pos2.keys())
    if len(common) < 2:
        return np.nan

    common = list(common)
    v1 = [pos1[x] for x in common]
    v2 = [pos2[x] for x in common]
    rho, _ = spearmanr(v1, v2)
    return rho

def pooled_kendall_tau(rows):
    """
    Treat each row as an independent ranking of k items.
    Aggregate concordant/discordant counts across all rows.
    For no ties:
      tau = (C - D) / (C + D)
    """
    C = 0
    D = 0

    for r1, r2 in rows:
        pos1 = ranks_as_positions(r1)
        pos2 = ranks_as_positions(r2)
        common = list(set(pos1.keys()) & set(pos2.keys()))
        if len(common) < 2:
            continue

        for a, b in combinations(common, 2):
            s1 = np.sign(pos1[a] - pos1[b])
            s2 = np.sign(pos2[a] - pos2[b])
            prod = s1 * s2
            if prod > 0:
                C += 1
            elif prod < 0:
                D += 1

    if C + D == 0:
        return np.nan
    return (C - D) / (C + D)

def same_item_set(a, b):
    """
    True if both are lists, contain no duplicates, and rank exactly the same set of items.
    This is the cleanest assumption for agreement metrics.
    """
    if a is None or b is None:
        return False
    if not isinstance(a, list) or not isinstance(b, list):
        return False
    if len(a) != len(set(a)) or len(b) != len(set(b)):
        return False
    return set(a) == set(b)

# =========================
# LOAD + PARSE
# =========================
df = pd.read_csv(CSV_PATH)

df["rank_a"] = df[COL_A].apply(parse_rank_list)
df["rank_b"] = df[COL_B].apply(parse_rank_list)

df["k_a"] = df["rank_a"].apply(lambda x: len(x) if isinstance(x, list) else 0)
df["k_b"] = df["rank_b"].apply(lambda x: len(x) if isinstance(x, list) else 0)

# Keep rows with k>=MIN_K in BOTH rankings and same ranked item set
mask_k = (df["k_a"] >= MIN_K) & (df["k_b"] >= MIN_K)
mask_same = df.apply(lambda r: same_item_set(r["rank_a"], r["rank_b"]), axis=1)

df_k = df[mask_k & mask_same].copy()

print("Rows total:", len(df))
print(f"Rows with k >= {MIN_K} in both:", mask_k.sum())
print(f"Rows with k >= {MIN_K} and same item set:", len(df_k))

# =========================
# COMPUTE PER-ROW AGREEMENT (k>=MIN_K)
# =========================
df_k["kendall_tau"] = df_k.apply(lambda r: kendall_tau_no_ties(r["rank_a"], r["rank_b"]), axis=1)
df_k["spearman_rho"] = df_k.apply(lambda r: spearman_rho_no_ties(r["rank_a"], r["rank_b"]), axis=1)

print("\n=== k >= {} results ===".format(MIN_K))
print("Rows:", len(df_k))
print("Valid tau rows:", df_k["kendall_tau"].notna().sum())
print("Mean Kendall tau:", df_k["kendall_tau"].mean())
print("Median Kendall tau:", df_k["kendall_tau"].median())
print("Mean Spearman rho:", df_k["spearman_rho"].mean())
print("Median Spearman rho:", df_k["spearman_rho"].median())

# =========================
# POOLED / GLOBAL TAU (k>=MIN_K)
# =========================
valid_pairs_k = df_k[["rank_a", "rank_b"]].itertuples(index=False, name=None)
global_tau_k = pooled_kendall_tau(list(valid_pairs_k))
print("Pooled/Global Kendall tau (k >= {}):".format(MIN_K), global_tau_k)

# =========================
# SAVE
# =========================
OUT_PATH = f"ira_rank_agreement_output_k{MIN_K}plus.csv"
df_k.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)

Rows total: 407
Rows with k >= 2 in both: 403
Rows with k >= 2 and same item set: 403

=== k >= 2 results ===
Rows: 403
Valid tau rows: 403
Mean Kendall tau: 0.9338296112489661
Median Kendall tau: 1.0
Mean Spearman rho: 0.9374689826302728
Median Spearman rho: 0.9999999999999999
Pooled/Global Kendall tau (k >= 2): 0.9312406576980568
Saved: ira_rank_agreement_output_k2plus.csv


In [14]:
print("df columns sample:", df.columns.tolist()[:40])


df columns sample: ['post_id', 'subreddit', 'title', 'author', 'score', 'num_comments_listed', 'op_replied', 'op_reply_count', 'created_utc', 'permalink', 'body', 'combined_text', 'bin01_comment', 'bin02_comment', 'bin03_comment', 'bin04_comment', 'bin05_comment', 'bin06_comment', 'bin07_comment', 'bin08_comment', 'bin09_comment', 'bin10_comment', 'Human_Ranking', 'LLM_Ranking', 'LLM_Ranking_2', 'LLM_Ranking_Error', 'LLM_Ranking_Valid', 'rank_a', 'rank_b', 'kendall_tau', 'spearman_rho']


In [15]:
import pandas as pd
import numpy as np

IN_PATH = "Ambivalent_Binning_with_Rankings_d0.csv"
OUT_PATH = "Ambivalent_Binning_with_Rankings_od0.csv"

df = pd.read_csv(IN_PATH)

def reverse_ranking(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return s
    txt = str(s).strip()
    if txt == "" or txt.lower() == "nan":
        return txt
    parts = [p.strip() for p in txt.split(",") if p.strip() != ""]
    parts_rev = list(reversed(parts))
    return ",".join(parts_rev)

df["Human_Ranking"] = df["Human_Ranking"].apply(reverse_ranking)

df.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)

# quick check
print(df[["post_id", "Human_Ranking"]].head(5).to_string(index=False))


Saved: Ambivalent_Binning_with_Rankings_od0.csv
post_id Human_Ranking
1hv4zdy           1,2
1bautv8           1,2
11f4o14           1,2
10olxi4           1,2
104hhnz           1,2


In [13]:
%pip -q install pingouin

You should consider upgrading via the '/home/ec2-user/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [14]:
import pandas as pd

CSV_PATH = "Ambivalent_Binning_with_Rankings_d0_Mistral.csv"
df = pd.read_csv(CSV_PATH)

# If you saved the ranked output already, use THAT instead:
# df = pd.read_csv("/mnt/data/Ambivalent_posts_with_top10_comments_with_rankings.csv")

df_out = df.copy()

In [15]:
df_out.columns = (
    df_out.columns.astype(str)
    .str.replace("\ufeff", "", regex=False)
    .str.strip()
)

required = ["post_id", "Human_Ranking", "LLM_Ranking"]
missing = [c for c in required if c not in df_out.columns]
print("Missing columns:", missing)

df_out[required].head(5)


Missing columns: []


,post_id,Human_Ranking,LLM_Ranking
0,1hv4zdy,"2,1","1,2"
1,1bautv8,"2,1","2,1"
2,11f4o14,"2,1","1,2"
3,10olxi4,"2,1","2,1"
4,104hhnz,"2,1","1,2"


In [16]:
import numpy as np

def parse_rank_vector(raw: str, n: int):
    """
    Parse rank-vector encoding of length n, e.g.:
      "1,1,3" or "1.0, 1.0, 3.0" or "[1, 1, 3]" or "Ranking: 1,1,3"
    Returns dict {item_index: rank_value} for item_index in 1..n.
    """
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        raise ValueError("Missing ranking")

    s = str(raw).strip()
    if s == "" or s.lower() == "nan":
        raise ValueError("Empty ranking")

    s = s.replace("Ranking:", "").strip()
    s = s.strip("[](){}")

    parts = [p.strip() for p in s.split(",") if p.strip() != ""]
    if len(parts) != n:
        raise ValueError(f"Expected {n} numbers but got {len(parts)} from '{raw}'")

    vals = [float(p) for p in parts]
    return {i: float(vals[i-1]) for i in range(1, n + 1)}

def rank_norm(rank_val: float, n: int) -> float:
    """
    Normalize ranks so different n are comparable across posts:
      best -> 0, worst -> 1
    """
    if n <= 1:
        return np.nan
    return (float(rank_val) - 1.0) / (n - 1.0)


In [17]:
use = df_out.dropna(subset=["post_id", "Human_Ranking", "LLM_Ranking"]).copy()

rows = []
bad_rows = 0

for _, r in use.iterrows():
    pid = str(r["post_id"])

    # n = number of ranks in Human_Ranking (since it's always "1,2,...,n")
    try:
        n = len([p for p in str(r["Human_Ranking"]).split(",") if p.strip() != ""])
        if n < 2:
            continue
    except Exception:
        bad_rows += 1
        continue

    try:
        hr = parse_rank_vector(r["Human_Ranking"], n)
        lr = parse_rank_vector(r["LLM_Ranking"], n)
    except Exception:
        bad_rows += 1
        continue

    for item in range(1, n + 1):
        target = f"{pid}_resp{item}"
        rows.append({"target": target, "rater": "human", "rating": rank_norm(hr[item], n)})
        rows.append({"target": target, "rater": "llm",   "rating": rank_norm(lr[item], n)})

long_df = pd.DataFrame(rows, columns=["target", "rater", "rating"])

print("Rows considered:", len(use))
print("Skipped rows (parse issues or n<2):", bad_rows)
print("long_df shape:", long_df.shape)
long_df.head(10)


Rows considered: 407
Skipped rows (parse issues or n<2): 6
long_df shape: (1854, 3)


,target,rater,rating
0,1hv4zdy_resp1,human,1.0
1,1hv4zdy_resp1,llm,0.0
2,1hv4zdy_resp2,human,0.0
3,1hv4zdy_resp2,llm,1.0
4,1bautv8_resp1,human,1.0
5,1bautv8_resp1,llm,1.0
6,1bautv8_resp2,human,0.0
7,1bautv8_resp2,llm,0.0
8,11f4o14_resp1,human,1.0
9,11f4o14_resp1,llm,0.0


In [18]:
import pingouin as pg

icc_tbl = pg.intraclass_corr(
    data=long_df,
    targets="target",
    raters="rater",
    ratings="rating"
)

icc_tbl


,Type,Description,ICC,F,df1,df2,pval,CI95%
0,ICC1,Single raters absolute,0.503694,3.029773,926,927,3.371313e-61,"[0.45, 0.55]"
1,ICC2,Single random raters,0.503653,3.028763,926,926,4.023965e-61,"[0.45, 0.55]"
2,ICC3,Single fixed raters,0.503570,3.028763,926,926,4.023965e-61,"[0.45, 0.55]"
3,ICC1k,Average raters absolute,0.669942,3.029773,926,927,3.371313e-61,"[0.62, 0.71]"
4,ICC2k,Average random raters,0.669906,3.028763,926,926,4.023965e-61,"[0.62, 0.71]"
5,ICC3k,Average fixed raters,0.669832,3.028763,926,926,4.023965e-61,"[0.62, 0.71]"


In [19]:
icc_tbl[icc_tbl["Type"].isin(["ICC2", "ICC3"])]


,Type,Description,ICC,F,df1,df2,pval,CI95%
1,ICC2,Single random raters,0.503653,3.028763,926,926,4.023965e-61,"[0.45, 0.55]"
2,ICC3,Single fixed raters,0.503570,3.028763,926,926,4.023965e-61,"[0.45, 0.55]"


In [20]:
icc2 = icc_tbl.loc[icc_tbl["Type"] == "ICC2"].iloc[0]
icc3 = icc_tbl.loc[icc_tbl["Type"] == "ICC3"].iloc[0]

print("Pooled ICC on normalized ranks (human vs LLM):\n")

print("ICC(2,1) [ICC2] — absolute agreement (two-way random effects):")
print("  ICC :", float(icc2["ICC"]))
print("  p   :", float(icc2["pval"]))
print("  CI95:", icc2["CI95%"])
print()

print("ICC(3,1) [ICC3] — consistency (two-way mixed effects):")
print("  ICC :", float(icc3["ICC"]))
print("  p   :", float(icc3["pval"]))
print("  CI95:", icc3["CI95%"])


Pooled ICC on normalized ranks (human vs LLM):

ICC(2,1) [ICC2] — absolute agreement (two-way random effects):
  ICC : 0.5036530919006583
  p   : 4.0239654293069206e-61
  CI95: [0.45 0.55]

ICC(3,1) [ICC3] — consistency (two-way mixed effects):
  ICC : 0.5035697056991164
  p   : 4.0239654293069206e-61
  CI95: [0.45 0.55]


In [26]:
icc2 = icc_tbl.loc[icc_tbl["Type"] == "ICC2"].iloc[0]
icc3 = icc_tbl.loc[icc_tbl["Type"] == "ICC3"].iloc[0]

print("Pooled ICC on normalized ranks (human vs LLM):\n")

print("ICC(2,1) [ICC2] — absolute agreement (two-way random effects):")
print("  ICC :", float(icc2["ICC"]))
print("  p   :", float(icc2["pval"]))
print("  CI95:", icc2["CI95%"])
print()

print("ICC(3,1) [ICC3] — consistency (two-way mixed effects):")
print("  ICC :", float(icc3["ICC"]))
print("  p   :", float(icc3["pval"]))
print("  CI95:", icc3["CI95%"])


Pooled ICC on normalized ranks (human vs LLM):

ICC(2,1) [ICC2] — absolute agreement (two-way random effects):
  ICC : 0.3417673778240041
  p   : 1.7161718298243544e-27
  CI95: [0.28 0.4 ]

ICC(3,1) [ICC3] — consistency (two-way mixed effects):
  ICC : 0.34152847752080934
  p   : 1.7161718298243544e-27
  CI95: [0.28 0.4 ]


In [49]:
icc2 = icc_tbl.loc[icc_tbl["Type"] == "ICC2"].iloc[0]
icc3 = icc_tbl.loc[icc_tbl["Type"] == "ICC3"].iloc[0]

print("Pooled ICC on normalized ranks (human vs LLM):\n")

print("ICC(2,1) [ICC2] — absolute agreement (two-way random effects):")
print("  ICC :", float(icc2["ICC"]))
print("  p   :", float(icc2["pval"]))
print("  CI95:", icc2["CI95%"])
print()

print("ICC(3,1) [ICC3] — consistency (two-way mixed effects):")
print("  ICC :", float(icc3["ICC"]))
print("  p   :", float(icc3["pval"]))
print("  CI95:", icc3["CI95%"])


Pooled ICC on normalized ranks (human vs LLM):

ICC(2,1) [ICC2] — absolute agreement (two-way random effects):
  ICC : 0.3422437485253651
  p   : 1.4409781146205245e-27
  CI95: [0.28 0.4 ]

ICC(3,1) [ICC3] — consistency (two-way mixed effects):
  ICC : 0.3420046882472267
  p   : 1.4409781146205245e-27
  CI95: [0.28 0.4 ]


In [21]:
pip install scipy

You should consider upgrading via the '/home/ec2-user/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [25]:
import re
import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr

IN_PATH = "Ambivalent_Binning_with_Rankings_d0_Mistral.csv"
df = pd.read_csv(IN_PATH)

print("Loaded:", IN_PATH)
print("Shape:", df.shape)

H_COL = "Human_Ranking"
L_COL = "LLM_Ranking"

def extract_ints(s: str):
    return [int(x) for x in re.findall(r"-?\d+", str(s))]

def parse_rankvec(s, n: int):
    if pd.isna(s):
        raise ValueError("Ranking is NaN/empty")
    nums = extract_ints(s)
    if len(nums) != n:
        raise ValueError(f"Expected {n} ranks, got {len(nums)} -> {nums}")
    if any(r <= 0 for r in nums):
        raise ValueError(f"Ranks must be positive integers -> {nums}")
    return nums

def infer_n_from_row(row):
    # Prefer n_comments_used if present
    n = row.get("n_comments_used", np.nan)
    if not pd.isna(n):
        try:
            n = int(float(n))
            if n > 0:
                return n
        except Exception:
            pass

    # fallback from human ranking length
    h = row.get(H_COL, None)
    if h is not None and not pd.isna(h):
        hn = len(extract_ints(h))
        if hn > 0:
            return hn

    # fallback from llm ranking length
    m = row.get(L_COL, None)
    if m is not None and not pd.isna(m):
        mn = len(extract_ints(m))
        if mn > 0:
            return mn

    raise ValueError("Could not infer n.")

def compute_ira_row(row):
    n = infer_n_from_row(row)

    human = parse_rankvec(row[H_COL], n)
    llm   = parse_rankvec(row[L_COL], n)

    tau, tau_p = kendalltau(human, llm, variant="b")  # tie-aware
    rho, rho_p = spearmanr(human, llm)                # tie-aware

    return pd.Series({
        "n_used_for_ira": n,
        "ira_kendall_tau_b": tau,
        "ira_kendall_p": tau_p,
        "ira_spearman_rho": rho,
        "ira_spearman_p": rho_p,
        "ira_valid": True,
        "ira_error": ""
    })

# 3) Compute IRA per row
required = {H_COL, L_COL}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

ira_out = []
for _, row in df.iterrows():
    try:
        ira_out.append(compute_ira_row(row))
    except Exception as e:
        ira_out.append(pd.Series({
            "n_used_for_ira": np.nan,
            "ira_kendall_tau_b": np.nan,
            "ira_kendall_p": np.nan,
            "ira_spearman_rho": np.nan,
            "ira_spearman_p": np.nan,
            "ira_valid": False,
            "ira_error": str(e)
        }))

ira_df = pd.DataFrame(ira_out)
df_with_ira = pd.concat([df.reset_index(drop=True), ira_df], axis=1)

# 4) Summary (overall IRA)
valid = df_with_ira["ira_valid"] == True

# Optional: if you have LLM_Ranking_Valid column, AND it exists, use it too
if "LLM_Ranking_Valid" in df_with_ira.columns:
    valid = valid & (df_with_ira["LLM_Ranking_Valid"] == True)

print("\nValid rows:", int(valid.sum()), "/", len(df_with_ira))
print("Mean Kendall Tau-b:", df_with_ira.loc[valid, "ira_kendall_tau_b"].mean())
print("Mean Spearman Rho:", df_with_ira.loc[valid, "ira_spearman_rho"].mean())

if (~valid).any():
    cols_to_show = [c for c in ["post_id", "n_comments_used", H_COL, L_COL, "ira_error"] if c in df_with_ira.columns]
    print("\nSample invalid rows:")
    print(df_with_ira.loc[~valid, cols_to_show].head(10).to_string(index=False))

# 5) Save
OUT_PATH = "Ambivalent_Binning_with_IRA_Mistral.csv"
df_with_ira.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


Loaded: Ambivalent_Binning_with_Rankings_d0_Mistral.csv
Shape: (407, 26)

Valid rows: 395 / 407
Mean Kendall Tau-b: 0.5380028129395218
Mean Spearman Rho: 0.5450972656035946

Sample invalid rows:
post_id Human_Ranking LLM_Ranking                            ira_error
1np7pss           2,1         2,3                                     
1barpvu           2,1           2       Expected 2 ranks, got 1 -> [2]
 wrbkr5           2,1           2       Expected 2 ranks, got 1 -> [2]
 w51ylq           2,1         2,3                                     
1fczliu         3,2,1         1,3    Expected 3 ranks, got 2 -> [1, 3]
1ouqac6           2,1           2       Expected 2 ranks, got 1 -> [2]
1ip49pg           2,1         2,3                                     
1h1sfht       4,3,2,1       4,2,1 Expected 4 ranks, got 3 -> [4, 2, 1]
 vpca35           2,1         2,3                                     
 qlphm1           2,1         2,3                                     

Saved: Ambivalent_Binni

In [28]:
import re
import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr

IN_PATH = "Ambivalent_Binning_with_Rankings_d0_4.csv"
df = pd.read_csv(IN_PATH)

print("Loaded:", IN_PATH)
print("Shape:", df.shape)

H_COL = "Human_Ranking"
L_COL = "LLM_Ranking"

def extract_ints(s: str):
    return [int(x) for x in re.findall(r"-?\d+", str(s))]

def parse_rankvec(s, n: int):
    if pd.isna(s):
        raise ValueError("Ranking is NaN/empty")
    nums = extract_ints(s)
    if len(nums) != n:
        raise ValueError(f"Expected {n} ranks, got {len(nums)} -> {nums}")
    if any(r <= 0 for r in nums):
        raise ValueError(f"Ranks must be positive integers -> {nums}")
    return nums

def infer_n_from_row(row):
    # Prefer n_comments_used if present
    n = row.get("n_comments_used", np.nan)
    if not pd.isna(n):
        try:
            n = int(float(n))
            if n > 0:
                return n
        except Exception:
            pass

    # fallback from human ranking length
    h = row.get(H_COL, None)
    if h is not None and not pd.isna(h):
        hn = len(extract_ints(h))
        if hn > 0:
            return hn

    # fallback from llm ranking length
    m = row.get(L_COL, None)
    if m is not None and not pd.isna(m):
        mn = len(extract_ints(m))
        if mn > 0:
            return mn

    raise ValueError("Could not infer n.")

def compute_ira_row(row):
    n = infer_n_from_row(row)

    human = parse_rankvec(row[H_COL], n)
    llm   = parse_rankvec(row[L_COL], n)

    tau, tau_p = kendalltau(human, llm, variant="b")  # tie-aware
    rho, rho_p = spearmanr(human, llm)                # tie-aware

    return pd.Series({
        "n_used_for_ira": n,
        "ira_kendall_tau_b": tau,
        "ira_kendall_p": tau_p,
        "ira_spearman_rho": rho,
        "ira_spearman_p": rho_p,
        "ira_valid": True,
        "ira_error": ""
    })

# 3) Compute IRA per row
required = {H_COL, L_COL}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

ira_out = []
for _, row in df.iterrows():
    try:
        ira_out.append(compute_ira_row(row))
    except Exception as e:
        ira_out.append(pd.Series({
            "n_used_for_ira": np.nan,
            "ira_kendall_tau_b": np.nan,
            "ira_kendall_p": np.nan,
            "ira_spearman_rho": np.nan,
            "ira_spearman_p": np.nan,
            "ira_valid": False,
            "ira_error": str(e)
        }))

ira_df = pd.DataFrame(ira_out)
df_with_ira = pd.concat([df.reset_index(drop=True), ira_df], axis=1)

# 4) Summary (overall IRA)
valid = df_with_ira["ira_valid"] == True

# Optional: if you have LLM_Ranking_Valid column, AND it exists, use it too
if "LLM_Ranking_Valid" in df_with_ira.columns:
    valid = valid & (df_with_ira["LLM_Ranking_Valid"] == True)

print("\nValid rows:", int(valid.sum()), "/", len(df_with_ira))
print("Mean Kendall Tau-b:", df_with_ira.loc[valid, "ira_kendall_tau_b"].mean())
print("Mean Spearman Rho:", df_with_ira.loc[valid, "ira_spearman_rho"].mean())

if (~valid).any():
    cols_to_show = [c for c in ["post_id", "n_comments_used", H_COL, L_COL, "ira_error"] if c in df_with_ira.columns]
    print("\nSample invalid rows:")
    print(df_with_ira.loc[~valid, cols_to_show].head(10).to_string(index=False))

# 5) Save
OUT_PATH = "Ambivalent_Binning_with_IRA.csv"
df_with_ira.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


Loaded: Ambivalent_Binning_with_Rankings_d0_4.csv
Shape: (407, 26)

Valid rows: 407 / 407
Mean Kendall Tau-b: 0.33360633360633357
Mean Spearman Rho: 0.3389892253528617

Saved: Ambivalent_Binning_with_IRA.csv


In [51]:
import re
import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr

IN_PATH = "Ambivalent_Binning_with_Rankings_d0_gpt-5.csv"
df = pd.read_csv(IN_PATH)

print("Loaded:", IN_PATH)
print("Shape:", df.shape)

H_COL = "Human_Ranking"
L_COL = "LLM_Ranking"

def extract_ints(s: str):
    return [int(x) for x in re.findall(r"-?\d+", str(s))]

def parse_rankvec(s, n: int):
    if pd.isna(s):
        raise ValueError("Ranking is NaN/empty")
    nums = extract_ints(s)
    if len(nums) != n:
        raise ValueError(f"Expected {n} ranks, got {len(nums)} -> {nums}")
    if any(r <= 0 for r in nums):
        raise ValueError(f"Ranks must be positive integers -> {nums}")
    return nums

def infer_n_from_row(row):
    # Prefer n_comments_used if present
    n = row.get("n_comments_used", np.nan)
    if not pd.isna(n):
        try:
            n = int(float(n))
            if n > 0:
                return n
        except Exception:
            pass

    # fallback from human ranking length
    h = row.get(H_COL, None)
    if h is not None and not pd.isna(h):
        hn = len(extract_ints(h))
        if hn > 0:
            return hn

    # fallback from llm ranking length
    m = row.get(L_COL, None)
    if m is not None and not pd.isna(m):
        mn = len(extract_ints(m))
        if mn > 0:
            return mn

    raise ValueError("Could not infer n.")

def compute_ira_row(row):
    n = infer_n_from_row(row)

    human = parse_rankvec(row[H_COL], n)
    llm   = parse_rankvec(row[L_COL], n)

    tau, tau_p = kendalltau(human, llm, variant="b")  # tie-aware
    rho, rho_p = spearmanr(human, llm)                # tie-aware

    return pd.Series({
        "n_used_for_ira": n,
        "ira_kendall_tau_b": tau,
        "ira_kendall_p": tau_p,
        "ira_spearman_rho": rho,
        "ira_spearman_p": rho_p,
        "ira_valid": True,
        "ira_error": ""
    })

# 3) Compute IRA per row
required = {H_COL, L_COL}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

ira_out = []
for _, row in df.iterrows():
    try:
        ira_out.append(compute_ira_row(row))
    except Exception as e:
        ira_out.append(pd.Series({
            "n_used_for_ira": np.nan,
            "ira_kendall_tau_b": np.nan,
            "ira_kendall_p": np.nan,
            "ira_spearman_rho": np.nan,
            "ira_spearman_p": np.nan,
            "ira_valid": False,
            "ira_error": str(e)
        }))

ira_df = pd.DataFrame(ira_out)
df_with_ira = pd.concat([df.reset_index(drop=True), ira_df], axis=1)

# 4) Summary (overall IRA)
valid = df_with_ira["ira_valid"] == True

# Optional: if you have LLM_Ranking_Valid column, AND it exists, use it too
if "LLM_Ranking_Valid" in df_with_ira.columns:
    valid = valid & (df_with_ira["LLM_Ranking_Valid"] == True)

print("\nValid rows:", int(valid.sum()), "/", len(df_with_ira))
print("Mean Kendall Tau-b:", df_with_ira.loc[valid, "ira_kendall_tau_b"].mean())
print("Mean Spearman Rho:", df_with_ira.loc[valid, "ira_spearman_rho"].mean())

if (~valid).any():
    cols_to_show = [c for c in ["post_id", "n_comments_used", H_COL, L_COL, "ira_error"] if c in df_with_ira.columns]
    print("\nSample invalid rows:")
    print(df_with_ira.loc[~valid, cols_to_show].head(10).to_string(index=False))

# 5) Save
OUT_PATH = "Ambivalent_Binning_with_IRA-gpt-5.csv"
df_with_ira.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


Loaded: Ambivalent_Binning_with_Rankings_d0_gpt-5.csv
Shape: (407, 26)

Valid rows: 407 / 407
Mean Kendall Tau-b: 0.33467103467103465
Mean Spearman Rho: 0.3406133997043089

Saved: Ambivalent_Binning_with_IRA-gpt-5.csv


In [25]:
import re
import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr

IN_PATH = "Ambivalent_Binning_with_Rankings_d0.csv"
df = pd.read_csv(IN_PATH)

print("Loaded:", IN_PATH)
print("Shape:", df.shape)

H_COL = "Human_Ranking"
L_COL = "LLM_Ranking"

def extract_ints(s: str):
    return [int(x) for x in re.findall(r"-?\d+", str(s))]

def parse_rankvec(s, n: int):
    if pd.isna(s):
        raise ValueError("Ranking is NaN/empty")
    nums = extract_ints(s)
    if len(nums) != n:
        raise ValueError(f"Expected {n} ranks, got {len(nums)} -> {nums}")
    if any(r <= 0 for r in nums):
        raise ValueError(f"Ranks must be positive integers -> {nums}")
    return nums

def infer_n_from_row(row):
    # Prefer n_comments_used if present
    n = row.get("n_comments_used", np.nan)
    if not pd.isna(n):
        try:
            n = int(float(n))
            if n > 0:
                return n
        except Exception:
            pass

    # fallback from human ranking length
    h = row.get(H_COL, None)
    if h is not None and not pd.isna(h):
        hn = len(extract_ints(h))
        if hn > 0:
            return hn

    # fallback from llm ranking length
    m = row.get(L_COL, None)
    if m is not None and not pd.isna(m):
        mn = len(extract_ints(m))
        if mn > 0:
            return mn

    raise ValueError("Could not infer n.")

def compute_ira_row(row):
    n = infer_n_from_row(row)

    human = parse_rankvec(row[H_COL], n)
    llm   = parse_rankvec(row[L_COL], n)

    tau, tau_p = kendalltau(human, llm, variant="b")  # tie-aware
    rho, rho_p = spearmanr(human, llm)                # tie-aware

    return pd.Series({
        "n_used_for_ira": n,
        "ira_kendall_tau_b": tau,
        "ira_kendall_p": tau_p,
        "ira_spearman_rho": rho,
        "ira_spearman_p": rho_p,
        "ira_valid": True,
        "ira_error": ""
    })

# 3) Compute IRA per row
required = {H_COL, L_COL}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

ira_out = []
for _, row in df.iterrows():
    try:
        ira_out.append(compute_ira_row(row))
    except Exception as e:
        ira_out.append(pd.Series({
            "n_used_for_ira": np.nan,
            "ira_kendall_tau_b": np.nan,
            "ira_kendall_p": np.nan,
            "ira_spearman_rho": np.nan,
            "ira_spearman_p": np.nan,
            "ira_valid": False,
            "ira_error": str(e)
        }))

ira_df = pd.DataFrame(ira_out)
df_with_ira = pd.concat([df.reset_index(drop=True), ira_df], axis=1)

# 4) Summary (overall IRA)
valid = df_with_ira["ira_valid"] == True

# Optional: if you have LLM_Ranking_Valid column, AND it exists, use it too
if "LLM_Ranking_Valid" in df_with_ira.columns:
    valid = valid & (df_with_ira["LLM_Ranking_Valid"] == True)

print("\nValid rows:", int(valid.sum()), "/", len(df_with_ira))
print("Mean Kendall Tau-b:", df_with_ira.loc[valid, "ira_kendall_tau_b"].mean())
print("Mean Spearman Rho:", df_with_ira.loc[valid, "ira_spearman_rho"].mean())

if (~valid).any():
    cols_to_show = [c for c in ["post_id", "n_comments_used", H_COL, L_COL, "ira_error"] if c in df_with_ira.columns]
    print("\nSample invalid rows:")
    print(df_with_ira.loc[~valid, cols_to_show].head(10).to_string(index=False))

# 5) Save
OUT_PATH = "Ambivalent_Binning_with_IRA.csv"
df_with_ira.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


Loaded: Ambivalent_Binning_with_Rankings_d0.csv
Shape: (407, 26)

Valid rows: 407 / 407
Mean Kendall Tau-b: 0.2983073983073983
Mean Spearman Rho: 0.3054075326802599

Saved: Ambivalent_Binning_with_IRA.csv


## IRA

In [26]:
pip install scipy

You should consider upgrading via the '/home/ec2-user/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [27]:
import re
import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr

# =========================
# 1) Load input
# =========================
IN_PATH = "../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_25_50.csv"

# If your file is tab-separated, switch to: pd.read_csv(IN_PATH, sep="\t")
df = pd.read_csv(IN_PATH)

print("Loaded:", IN_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# =========================
# 2) Helpers
# =========================
def extract_ints(s: str):
    """Extract integers from a string like '1,1,3' or 'Ranking: 1, 1, 3' or '[1,1,3]'."""
    return [int(x) for x in re.findall(r"-?\d+", str(s))]

def parse_rankvec(s, n: int):
    """
    Parse a per-comment rank vector from a cell like:
      "1,1,3" or "1, 1, 3" or "[1,1,3]" or "Ranking: 1, 1, 3"
    Returns: list[int] of length n
    """
    if pd.isna(s):
        raise ValueError("Ranking is NaN")

    nums = extract_ints(s)

    if len(nums) != n:
        raise ValueError(f"Expected {n} ranks, got {len(nums)} -> {nums}")

    if any(r <= 0 for r in nums):
        raise ValueError(f"Ranks must be positive integers -> {nums}")

    return nums

def infer_n_from_rankings(row):
    """
    Prefer n_comments_used if present/valid, otherwise infer from Human_Rankings length,
    otherwise infer from LLM_Rankings length.
    """
    n = row.get("n_comments_used", np.nan)

    if not pd.isna(n):
        try:
            n = int(n)
            if n > 0:
                return n
        except Exception:
            pass

    # fallback: infer from human ranking length
    h = row.get("Human_Rankings", None)
    if h is not None and not pd.isna(h):
        hn = len(extract_ints(h))
        if hn > 0:
            return hn

    # fallback: infer from llm ranking length
    m = row.get("LLM_Rankings", None)
    if m is not None and not pd.isna(m):
        mn = len(extract_ints(m))
        if mn > 0:
            return mn

    raise ValueError("Could not infer n (n_comments_used missing/invalid and rankings empty).")

def compute_ira_row(row):
    n = infer_n_from_rankings(row)

    human = parse_rankvec(row["Human_Rankings"], n)
    llm   = parse_rankvec(row["LLM_Rankings"], n)

    # Kendall tau-b handles ties properly
    tau, tau_p = kendalltau(human, llm, variant="b")

    # Spearman rho (also supports ties)
    rho, rho_p = spearmanr(human, llm)

    return pd.Series({
        "n_used_for_ira": n,
        "ira_kendall_tau_b": tau,
        "ira_kendall_p": tau_p,
        "ira_spearman_rho": rho,
        "ira_spearman_p": rho_p,
        "ira_valid": True,
        "ira_error": ""
    })

# =========================
# 3) Compute IRA per row
# =========================
required = {"Human_Rankings", "LLM_Rankings"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

ira_out = []
for _, row in df.iterrows():
    try:
        ira_out.append(compute_ira_row(row))
    except Exception as e:
        ira_out.append(pd.Series({
            "n_used_for_ira": np.nan,
            "ira_kendall_tau_b": np.nan,
            "ira_kendall_p": np.nan,
            "ira_spearman_rho": np.nan,
            "ira_spearman_p": np.nan,
            "ira_valid": False,
            "ira_error": str(e)
        }))

ira_df = pd.DataFrame(ira_out)
df_with_ira = pd.concat([df.reset_index(drop=True), ira_df], axis=1)

# =========================
# 4) Summary (overall IRA)
# =========================
valid = df_with_ira["ira_valid"] == True
print("\nValid rows:", int(valid.sum()), "/", len(df_with_ira))
print("Mean Kendall Tau-b (IRA):", df_with_ira.loc[valid, "ira_kendall_tau_b"].mean())
print("Mean Spearman Rho:", df_with_ira.loc[valid, "ira_spearman_rho"].mean())

# Show a few failures (if any)
if (~valid).any():
    print("\nSample invalid rows:")
    cols_to_show = [c for c in ["post_id", "n_comments_used", "Human_Rankings", "LLM_Rankings", "ira_error"] if c in df_with_ira.columns]
    print(df_with_ira.loc[~valid, cols_to_show].head(10).to_string(index=False))

# =========================
# 5) Save
# =========================
OUT_PATH = "../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_25_50_with_IRA.csv"
df_with_ira.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


Loaded: ../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_25_50.csv
Shape: (195, 31)
Columns: ['post_id', 'subreddit', 'title', 'author', 'score', 'num_comments_listed', 'op_replied', 'op_reply_count', 'created_utc', 'permalink', 'body', 'combined_text', 'matched_ambivalent_phrases', 'matched_keywords', 'TC1', 'TC2', 'TC3', 'TC4', 'TC5', 'TC6', 'TC7', 'TC8', 'TC9', 'TC10', 'row_index', 'n_comments_used', 'ranking_raw', 'ranking_valid', 'ranking_error', 'Human_Rankings', 'LLM_Rankings']

Valid rows: 195 / 195
Mean Kendall Tau-b (IRA): 0.1967844147521091
Mean Spearman Rho: 0.19833756375580863

Saved: ../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_25_50_with_IRA.csv


In [49]:
import re
import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr

# =========================
# 1) Load input
# =========================
IN_PATH = "../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_50_75.csv"

# If your file is tab-separated, switch to: pd.read_csv(IN_PATH, sep="\t")
df = pd.read_csv(IN_PATH)

print("Loaded:", IN_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# =========================
# 2) Helpers
# =========================
def extract_ints(s: str):
    """Extract integers from a string like '1,1,3' or 'Ranking: 1, 1, 3' or '[1,1,3]'."""
    return [int(x) for x in re.findall(r"-?\d+", str(s))]

def parse_rankvec(s, n: int):
    """
    Parse a per-comment rank vector from a cell like:
      "1,1,3" or "1, 1, 3" or "[1,1,3]" or "Ranking: 1, 1, 3"
    Returns: list[int] of length n
    """
    if pd.isna(s):
        raise ValueError("Ranking is NaN")

    nums = extract_ints(s)

    if len(nums) != n:
        raise ValueError(f"Expected {n} ranks, got {len(nums)} -> {nums}")

    if any(r <= 0 for r in nums):
        raise ValueError(f"Ranks must be positive integers -> {nums}")

    return nums

def infer_n_from_rankings(row):
    """
    Prefer n_comments_used if present/valid, otherwise infer from Human_Rankings length,
    otherwise infer from LLM_Rankings length.
    """
    n = row.get("n_comments_used", np.nan)

    if not pd.isna(n):
        try:
            n = int(n)
            if n > 0:
                return n
        except Exception:
            pass

    # fallback: infer from human ranking length
    h = row.get("Human_Rankings", None)
    if h is not None and not pd.isna(h):
        hn = len(extract_ints(h))
        if hn > 0:
            return hn

    # fallback: infer from llm ranking length
    m = row.get("LLM_Rankings", None)
    if m is not None and not pd.isna(m):
        mn = len(extract_ints(m))
        if mn > 0:
            return mn

    raise ValueError("Could not infer n (n_comments_used missing/invalid and rankings empty).")

def compute_ira_row(row):
    n = infer_n_from_rankings(row)

    human = parse_rankvec(row["Human_Rankings"], n)
    llm   = parse_rankvec(row["LLM_Rankings"], n)

    # Kendall tau-b handles ties properly
    tau, tau_p = kendalltau(human, llm, variant="b")

    # Spearman rho (also supports ties)
    rho, rho_p = spearmanr(human, llm)

    return pd.Series({
        "n_used_for_ira": n,
        "ira_kendall_tau_b": tau,
        "ira_kendall_p": tau_p,
        "ira_spearman_rho": rho,
        "ira_spearman_p": rho_p,
        "ira_valid": True,
        "ira_error": ""
    })

# =========================
# 3) Compute IRA per row
# =========================
required = {"Human_Rankings", "LLM_Rankings"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

ira_out = []
for _, row in df.iterrows():
    try:
        ira_out.append(compute_ira_row(row))
    except Exception as e:
        ira_out.append(pd.Series({
            "n_used_for_ira": np.nan,
            "ira_kendall_tau_b": np.nan,
            "ira_kendall_p": np.nan,
            "ira_spearman_rho": np.nan,
            "ira_spearman_p": np.nan,
            "ira_valid": False,
            "ira_error": str(e)
        }))

ira_df = pd.DataFrame(ira_out)
df_with_ira = pd.concat([df.reset_index(drop=True), ira_df], axis=1)

# =========================
# 4) Summary (overall IRA)
# =========================
valid = df_with_ira["ira_valid"] == True
print("\nValid rows:", int(valid.sum()), "/", len(df_with_ira))
print("Mean Kendall Tau-b (IRA):", df_with_ira.loc[valid, "ira_kendall_tau_b"].mean())
print("Mean Spearman Rho:", df_with_ira.loc[valid, "ira_spearman_rho"].mean())

# Show a few failures (if any)
if (~valid).any():
    print("\nSample invalid rows:")
    cols_to_show = [c for c in ["post_id", "n_comments_used", "Human_Rankings", "LLM_Rankings", "ira_error"] if c in df_with_ira.columns]
    print(df_with_ira.loc[~valid, cols_to_show].head(10).to_string(index=False))

# =========================
# 5) Save
# =========================
OUT_PATH = "../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_50_75_with_IRA.csv"
df_with_ira.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


Loaded: ../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_50_75.csv
Shape: (194, 31)
Columns: ['post_id', 'subreddit', 'title', 'author', 'score', 'num_comments_listed', 'op_replied', 'op_reply_count', 'created_utc', 'permalink', 'body', 'combined_text', 'matched_ambivalent_phrases', 'matched_keywords', 'TC1', 'TC2', 'TC3', 'TC4', 'TC5', 'TC6', 'TC7', 'TC8', 'TC9', 'TC10', 'row_index', 'n_comments_used', 'ranking_raw', 'ranking_valid', 'ranking_error', 'Human_Rankings', 'LLM_Rankings']

Valid rows: 194 / 194
Mean Kendall Tau-b (IRA): 0.2660290288746958
Mean Spearman Rho: 0.290542857411822

Saved: ../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_50_75_with_IRA.csv


In [50]:
import re
import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr

# =========================
# 1) Load input
# =========================
IN_PATH = "../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_75_100.csv"

# If your file is tab-separated, switch to: pd.read_csv(IN_PATH, sep="\t")
df = pd.read_csv(IN_PATH)

print("Loaded:", IN_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# =========================
# 2) Helpers
# =========================
def extract_ints(s: str):
    """Extract integers from a string like '1,1,3' or 'Ranking: 1, 1, 3' or '[1,1,3]'."""
    return [int(x) for x in re.findall(r"-?\d+", str(s))]

def parse_rankvec(s, n: int):
    """
    Parse a per-comment rank vector from a cell like:
      "1,1,3" or "1, 1, 3" or "[1,1,3]" or "Ranking: 1, 1, 3"
    Returns: list[int] of length n
    """
    if pd.isna(s):
        raise ValueError("Ranking is NaN")

    nums = extract_ints(s)

    if len(nums) != n:
        raise ValueError(f"Expected {n} ranks, got {len(nums)} -> {nums}")

    if any(r <= 0 for r in nums):
        raise ValueError(f"Ranks must be positive integers -> {nums}")

    return nums

def infer_n_from_rankings(row):
    """
    Prefer n_comments_used if present/valid, otherwise infer from Human_Rankings length,
    otherwise infer from LLM_Rankings length.
    """
    n = row.get("n_comments_used", np.nan)

    if not pd.isna(n):
        try:
            n = int(n)
            if n > 0:
                return n
        except Exception:
            pass

    # fallback: infer from human ranking length
    h = row.get("Human_Rankings", None)
    if h is not None and not pd.isna(h):
        hn = len(extract_ints(h))
        if hn > 0:
            return hn

    # fallback: infer from llm ranking length
    m = row.get("LLM_Rankings", None)
    if m is not None and not pd.isna(m):
        mn = len(extract_ints(m))
        if mn > 0:
            return mn

    raise ValueError("Could not infer n (n_comments_used missing/invalid and rankings empty).")

def compute_ira_row(row):
    n = infer_n_from_rankings(row)

    human = parse_rankvec(row["Human_Rankings"], n)
    llm   = parse_rankvec(row["LLM_Rankings"], n)

    # Kendall tau-b handles ties properly
    tau, tau_p = kendalltau(human, llm, variant="b")

    # Spearman rho (also supports ties)
    rho, rho_p = spearmanr(human, llm)

    return pd.Series({
        "n_used_for_ira": n,
        "ira_kendall_tau_b": tau,
        "ira_kendall_p": tau_p,
        "ira_spearman_rho": rho,
        "ira_spearman_p": rho_p,
        "ira_valid": True,
        "ira_error": ""
    })

# =========================
# 3) Compute IRA per row
# =========================
required = {"Human_Rankings", "LLM_Rankings"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

ira_out = []
for _, row in df.iterrows():
    try:
        ira_out.append(compute_ira_row(row))
    except Exception as e:
        ira_out.append(pd.Series({
            "n_used_for_ira": np.nan,
            "ira_kendall_tau_b": np.nan,
            "ira_kendall_p": np.nan,
            "ira_spearman_rho": np.nan,
            "ira_spearman_p": np.nan,
            "ira_valid": False,
            "ira_error": str(e)
        }))

ira_df = pd.DataFrame(ira_out)
df_with_ira = pd.concat([df.reset_index(drop=True), ira_df], axis=1)

# =========================
# 4) Summary (overall IRA)
# =========================
valid = df_with_ira["ira_valid"] == True
print("\nValid rows:", int(valid.sum()), "/", len(df_with_ira))
print("Mean Kendall Tau-b (IRA):", df_with_ira.loc[valid, "ira_kendall_tau_b"].mean())
print("Mean Spearman Rho:", df_with_ira.loc[valid, "ira_spearman_rho"].mean())

# Show a few failures (if any)
if (~valid).any():
    print("\nSample invalid rows:")
    cols_to_show = [c for c in ["post_id", "n_comments_used", "Human_Rankings", "LLM_Rankings", "ira_error"] if c in df_with_ira.columns]
    print(df_with_ira.loc[~valid, cols_to_show].head(10).to_string(index=False))

# =========================
# 5) Save
# =========================
OUT_PATH = "../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_75_100_with_IRA.csv"
df_with_ira.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


Loaded: ../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_75_100.csv
Shape: (160, 31)
Columns: ['post_id', 'subreddit', 'title', 'author', 'score', 'num_comments_listed', 'op_replied', 'op_reply_count', 'created_utc', 'permalink', 'body', 'combined_text', 'matched_ambivalent_phrases', 'matched_keywords', 'TC1', 'TC2', 'TC3', 'TC4', 'TC5', 'TC6', 'TC7', 'TC8', 'TC9', 'TC10', 'row_index', 'n_comments_used', 'ranking_raw', 'ranking_valid', 'ranking_error', 'Human_Rankings', 'LLM_Rankings']

Valid rows: 159 / 160
Mean Kendall Tau-b (IRA): 0.37701113492725724
Mean Spearman Rho: 0.4281988516911145

Sample invalid rows:
post_id  n_comments_used Human_Rankings LLM_Rankings                               ira_error
11t1ste                3          1,2,2      2,3,3,1 Expected 3 ranks, got 4 -> [2, 3, 3, 1]

Saved: ../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_75_100_with_IRA.csv


## Infra class correlation

In [17]:
%pip -q install pingouin

You should consider upgrading via the '/home/ec2-user/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [43]:
import pandas as pd

CSV_PATH = "../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_25_50_with_IRA.csv"
df = pd.read_csv(CSV_PATH)

# If you saved the ranked output already, use THAT instead:
# df = pd.read_csv("/mnt/data/Ambivalent_posts_with_top10_comments_with_rankings.csv")

df_out = df.copy()

In [44]:
df_out.columns = (
    df_out.columns.astype(str)
    .str.replace("\ufeff", "", regex=False)
    .str.strip()
)

required = ["post_id", "Human_Rankings", "LLM_Rankings", "n_used_for_ira"]
missing = [c for c in required if c not in df_out.columns]
print("Missing columns:", missing)

df_out[required].head(5)


Missing columns: []


,post_id,Human_Rankings,LLM_Rankings,n_used_for_ira
0,1hv4zdy,"1,1,3","3,2,1",3
1,11f4o14,"1,2","2,1",2
2,xyf4l9,"1,1,3","2,1,3",3
3,pihi7p,"1,1,3,3","1,4,3,2",4
4,1ofr7sg,"1,2","2,1",2


In [45]:
def parse_rank_vector(raw: str, n: int):
    """
    Parse rank-vector encoding of length n, e.g.:
      "1,1,3" or "1.0, 1.0, 3.0" or "[1, 1, 3]"
    Returns dict {item_index: rank_value} for item_index in 1..n.
    """
    s = str(raw).strip()
    s = s.replace("Ranking:", "").strip()
    s = s.strip("[](){}")

    parts = [p.strip() for p in s.split(",") if p.strip() != ""]
    if len(parts) != n:
        raise ValueError(f"Expected {n} numbers but got {len(parts)} from '{raw}'")

    vals = [float(p) for p in parts]
    return {i: float(vals[i-1]) for i in range(1, n + 1)}

def rank_norm(rank_val: float, n: int) -> float:
    """
    Normalize ranks so different n are comparable across posts:
      best -> 0, worst -> 1
    """
    if n <= 1:
        return np.nan
    return (float(rank_val) - 1.0) / (n - 1.0)


In [46]:
use = df_out.dropna(subset=["post_id", "Human_Rankings", "LLM_Rankings", "n_used_for_ira"]).copy()

rows = []
bad_rows = 0

for _, r in use.iterrows():
    pid = str(r["post_id"])
    n = int(float(r["n_used_for_ira"]))  # robust if stored as 3.0

    try:
        hr = parse_rank_vector(r["Human_Rankings"], n)
        lr = parse_rank_vector(r["LLM_Rankings"], n)
    except Exception:
        bad_rows += 1
        continue

    for item in range(1, n + 1):
        target = f"{pid}_resp{item}"
        rows.append({"target": target, "rater": "human", "rating": rank_norm(hr[item], n)})
        rows.append({"target": target, "rater": "llm",   "rating": rank_norm(lr[item], n)})

long_df = pd.DataFrame(rows, columns=["target", "rater", "rating"])

print("Rows used:", len(use))
print("Skipped rows (parse issues):", bad_rows)
print("long_df shape:", long_df.shape)
long_df.head(10)


Rows used: 195
Skipped rows (parse issues): 0
long_df shape: (1006, 3)


,target,rater,rating
0,1hv4zdy_resp1,human,0.0
1,1hv4zdy_resp1,llm,1.0
2,1hv4zdy_resp2,human,0.0
3,1hv4zdy_resp2,llm,0.5
4,1hv4zdy_resp3,human,1.0
5,1hv4zdy_resp3,llm,0.0
6,11f4o14_resp1,human,0.0
7,11f4o14_resp1,llm,1.0
8,11f4o14_resp2,human,1.0
9,11f4o14_resp2,llm,0.0


In [47]:
icc_tbl = pg.intraclass_corr(
    data=long_df,
    targets="target",
    raters="rater",
    ratings="rating"
)

icc_tbl


,Type,Description,ICC,F,df1,df2,pval,CI95%
0,ICC1,Single raters absolute,0.135662,1.313909,388,389,0.003628,"[0.04, 0.23]"
1,ICC2,Single random raters,0.145682,1.350563,388,388,0.001577,"[0.05, 0.24]"
2,ICC3,Single fixed raters,0.149140,1.350563,388,388,0.001577,"[0.05, 0.24]"
3,ICC1k,Average raters absolute,0.238912,1.313909,388,389,0.003628,"[0.07, 0.38]"
4,ICC2k,Average random raters,0.254315,1.350563,388,388,0.001577,"[0.09, 0.39]"
5,ICC3k,Average fixed raters,0.259568,1.350563,388,388,0.001577,"[0.1, 0.39]"


In [48]:
icc_tbl[icc_tbl["Type"].isin(["ICC2", "ICC3"])]


,Type,Description,ICC,F,df1,df2,pval,CI95%
1,ICC2,Single random raters,0.145682,1.350563,388,388,0.001577,"[0.05, 0.24]"
2,ICC3,Single fixed raters,0.149140,1.350563,388,388,0.001577,"[0.05, 0.24]"


In [49]:
icc2 = icc_tbl.loc[icc_tbl["Type"] == "ICC2"].iloc[0]
icc3 = icc_tbl.loc[icc_tbl["Type"] == "ICC3"].iloc[0]

print("ICC ranks for 25-50%:")

print("Pooled ICC(2,1) [ICC2] absolute agreement on normalized ranks:")
print("  ICC :", float(icc2["ICC"]))
print("  p   :", float(icc2["pval"]))
print("  CI95:", icc2["CI95%"])
print()

print("Pooled ICC(3,1) [ICC3] absolute agreement on normalized ranks:")
print("  ICC :", float(icc3["ICC"]))
print("  p   :", float(icc3["pval"]))
print("  CI95:", icc3["CI95%"])


ICC ranks for 25-50%:
Pooled ICC(2,1) [ICC2] absolute agreement on normalized ranks:
  ICC : 0.14568206052529056
  p   : 0.0015768033972234402
  CI95: [0.05 0.24]

Pooled ICC(3,1) [ICC3] absolute agreement on normalized ranks:
  ICC : 0.14914004819352575
  p   : 0.0015768033972234402
  CI95: [0.05 0.24]


In [34]:
icc2 = icc_tbl.loc[icc_tbl["Type"] == "ICC2"].iloc[0]
icc3 = icc_tbl.loc[icc_tbl["Type"] == "ICC3"].iloc[0]

print("ICC ranks for 50-75%:")

print("Pooled ICC(2,1) [ICC2] absolute agreement on normalized ranks:")
print("  ICC :", float(icc2["ICC"]))
print("  p   :", float(icc2["pval"]))
print("  CI95:", icc2["CI95%"])
print()

print("Pooled ICC(3,1) [ICC3] absolute agreement on normalized ranks:")
print("  ICC :", float(icc3["ICC"]))
print("  p   :", float(icc3["pval"]))
print("  CI95:", icc3["CI95%"])


ICC ranks for 50-75%:
Pooled ICC(2,1) [ICC2] absolute agreement on normalized ranks:
  ICC : 0.30941422850865863
  p   : 3.124683562865803e-14
  CI95: [0.23 0.38]

Pooled ICC(3,1) [ICC3] absolute agreement on normalized ranks:
  ICC : 0.31421756930548084
  p   : 3.124683562865803e-14
  CI95: [0.24 0.39]


In [42]:
icc2 = icc_tbl.loc[icc_tbl["Type"] == "ICC2"].iloc[0]
icc3 = icc_tbl.loc[icc_tbl["Type"] == "ICC3"].iloc[0]

print("ICC ranks for 75-100%:")

print("Pooled ICC(2,1) [ICC2] absolute agreement on normalized ranks:")
print("  ICC :", float(icc2["ICC"]))
print("  p   :", float(icc2["pval"]))
print("  CI95:", icc2["CI95%"])
print()

print("Pooled ICC(3,1) [ICC3] absolute agreement on normalized ranks:")
print("  ICC :", float(icc3["ICC"]))
print("  p   :", float(icc3["pval"]))
print("  CI95:", icc3["CI95%"])


ICC ranks for 75-100%:
Pooled ICC(2,1) [ICC2] absolute agreement on normalized ranks:
  ICC : 0.40158314445410276
  p   : 3.4394248315387075e-30
  CI95: [0.34 0.46]

Pooled ICC(3,1) [ICC3] absolute agreement on normalized ranks:
  ICC : 0.40330914499152115
  p   : 3.4394248315387075e-30
  CI95: [0.34 0.46]


In [16]:
import numpy as np
import pandas as pd

def parse_rank_vector_if_possible(raw: str, n: int):
    """
    Try to interpret raw as a rank-vector of length n.
    Supports:
      "1,1,3"
      "1.0, 1.0, 3.0"
      "[1, 1, 3]"
      "(1,1,3)"
    """
    s = str(raw).strip()

    # remove common wrappers
    s = s.replace("Ranking:", "").strip()
    s = s.strip("[](){}")

    # split on commas
    parts = [p.strip() for p in s.split(",") if p.strip() != ""]
    if len(parts) != n:
        return None

    # allow floats like "1.0"
    vals = []
    for p in parts:
        try:
            vals.append(float(p))
        except:
            return None

    # return as ranks per item index 1..n
    return {i: float(vals[i-1]) for i in range(1, n+1)}

def parse_ordered_ids(raw: str, n: int):
    """
    Ordered IDs format like:
      "3=1, 2"
      "3,2,1"
    Returns ranks per item index 1..n.
    """
    s = str(raw).strip()
    s = s.replace("Ranking:", "").strip()

    chunks = [c.strip() for c in s.split(",") if c.strip()]
    if not chunks:
        raise ValueError("empty ranking")

    groups = []
    for c in chunks:
        if "=" in c:
            tied = [int(x) for x in re.findall(r"\d+", c)]
            if tied:
                groups.append(tied)
        else:
            nums = [int(x) for x in re.findall(r"\d+", c)]
            # sequential items
            for num in nums:
                groups.append([num])

    ranks = {}
    current_rank = 1
    for tied in groups:
        k = len(tied)
        avg_rank = (current_rank + (current_rank + k - 1)) / 2.0
        for item in tied:
            ranks[item] = avg_rank
        current_rank += k

    if set(ranks.keys()) != set(range(1, n + 1)):
        raise ValueError(f"ordered ids does not cover 1..{n}. keys={sorted(ranks.keys())}")

    return ranks

def parse_ranks_any_v2(raw: str, n: int):
    # 1) Try rank-vector
    rv = parse_rank_vector_if_possible(raw, n)
    if rv is not None:
        return rv
    # 2) Fall back to ordered IDs
    return parse_ordered_ids(raw, n)

def rank_norm(rank_val: float, n: int) -> float:
    # best -> 0, worst -> 1
    if n <= 1:
        return np.nan
    return (float(rank_val) - 1.0) / (n - 1.0)

rows = []
bad_rows = 0
bad_examples = []

for _, r in use.iterrows():
    pid = str(r["post_id"])
    n = int(float(r["n_used_for_ira"]))  # handles 3.0 safely

    try:
        hr = parse_ranks_any_v2(r["Human_Rankings"], n)
        lr = parse_ranks_any_v2(r["LLM_Rankings"], n)
    except Exception as e:
        bad_rows += 1
        if len(bad_examples) < 5:
            bad_examples.append((pid, n, repr(r["Human_Rankings"]), repr(r["LLM_Rankings"]), str(e)))
        continue

    for item in range(1, n + 1):
        target = f"{pid}_resp{item}"
        rows.append({"target": target, "rater": "human", "rating": rank_norm(hr[item], n)})
        rows.append({"target": target, "rater": "llm",   "rating": rank_norm(lr[item], n)})

long_df = pd.DataFrame(rows, columns=["target", "rater", "rating"])

print("Total rows in use:", len(use))
print("Skipped rows (unparseable):", bad_rows)
print("long_df shape:", long_df.shape)
print("Some skipped examples:", bad_examples[:3])

long_df.head(10)


Total rows in use: 195
Skipped rows (unparseable): 0
long_df shape: (1006, 3)
Some skipped examples: []


,target,rater,rating
0,1hv4zdy_resp1,human,0.0
1,1hv4zdy_resp1,llm,1.0
2,1hv4zdy_resp2,human,0.0
3,1hv4zdy_resp2,llm,0.5
4,1hv4zdy_resp3,human,1.0
5,1hv4zdy_resp3,llm,0.0
6,11f4o14_resp1,human,0.0
7,11f4o14_resp1,llm,1.0
8,11f4o14_resp2,human,1.0
9,11f4o14_resp2,llm,0.0
